# 04 · Representations (first part: prototypes)

## Purpose
Prototype the image-like representations for the deep models before any full build, and compare the candidates so
Bhuvesh can choose one: accelerometer **order-spectrograms** on a fixed order axis, and a **microphone log-Mel** whose
frames are aligned in time with them so the two can be stacked. Nothing here builds a full cache or freezes a shape.

**Exploratory subset:** `split_closed` TRAIN ∩ the 2B.1 development pool, class-stratified, about 10 % (70 recordings).
Recordings at 40 Hz and above are reserved, so every 60 Hz example is synthetic.

## Inputs
`configs/base.yaml` (section `spectro`), the caches of `scripts/08_representations.py` (subset images, resolution,
sharpness, microphone diagnostics, candidate summary) and `data/raw_npy/` for the synthetic and burst examples.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code> and the prototype outputs, and prints the fixed order axis and each candidate's settings.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.config import load_config
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, cache_file, data_dir
from ml.spectro import order_axis, recording_representation

cfg = load_config(DEFAULT_CONFIG)
pre = cfg.preprocess
sc = pre.spectro
RES = SYSTEM_CODE_DIR / "results" / "representations"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
orders = order_axis(pre)
subs = {name: np.load(cache_file(f"08_subset_{name}", pre, ext="npz"), allow_pickle=True) for name in sc.candidates}
summary = json.loads((RES / "candidates_summary.json").read_text())
print(f"order axis: {sc.n_orders} log-spaced bins, {sc.order_min}–{sc.order_max} orders; declared speed ceiling "
      f"{sc.speed_ceiling_hz} Hz → {sc.order_max * sc.speed_ceiling_hz / 1e3:.1f} kHz < {pre.lowpass.pass_hz / 1e3} kHz pass-band edge")
pd.DataFrame({n: c.model_dump() for n, c in sc.candidates.items()}).T

## 1. The candidates
Frames: T per 1 s window, centred at `window_start + (i + 0.5) · 1 s / T`, cut from the whole cleaned recording
(reflect-padded only at the recording's ends). Spectra are mapped onto the 128 order bins by the **maximum over each
bin's span**, so a line keeps its peak amplitude wherever it falls; bins that hold no spectral point (low orders at low
speed) are interpolated. Zero-padding interpolates only; it adds no resolution.

| | analysis window | FFT length | window function | hop / T | order mapping |
| --- | --- | --- | --- | --- | --- |
| **A** STFT (the plan's) | 8,192 samples = 0.164 s | 16,384 | Hann | 40 ms / T = 25 | window's speed |
| **B** longer STFT | 32,768 samples = 0.655 s | 65,536 | Hann | 100 ms / T = 10 | window's speed |
| **C** order-domain | 8 revolutions (0.13–0.68 s over 62–11.8 Hz) | 8 rev × 512 × 2 | Hann | 40 ms / T = 25 | resampled in angle |

- **A** has 6.10 Hz bin spacing: 0.51 orders at 12 Hz, 0.20 at 30 Hz, 0.10 at 60 Hz.
- **B** has 1.53 Hz spacing (0.127 / 0.051 / 0.025 orders) at the cost of **time resolution**: each frame spans 0.655 s of
  the 1 s window, so neighbouring frames overlap by 85 %, and fast changes are smeared.
- **C** resamples each frame to 512 samples per revolution along the shaft angle (speed profile interpolated between the
  window speeds; linear interpolation), after a speed-dependent Kaiser anti-alias low-pass at 205 orders × speed whenever
  that is below the 9.8 kHz band limit. Its order resolution is 1/8 order at every speed; its **time** resolution varies
  with speed (a frame lasts 8 revolutions).

## 2. Resolution at low orders
Synthetic lines of equal amplitude at 0.5×, 1×, 2× and 3× (underhang radial) at 12, 30 and 60 Hz, time-averaged.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds synthetic recordings with equal lines at 0.5, 1, 2 and 3 times the shaft speed at 12, 30 and 60 Hz, runs each candidate on them, and plots the time-averaged order profile, so it is visible which candidate separates neighbouring low orders.</div>

In [ ]:
def synthetic(fr, lines=(0.5, 1, 2, 3), amp=0.1):
    t = np.arange(pre.n_rows) / pre.fs_hz
    x = 1e-3 * np.random.default_rng(0).normal(size=(pre.n_rows, 8))
    x[:, 0] = np.where((fr * t) % 1.0 < 0.12, 4.6, -0.7)
    for L in lines:
        x[:, pre.channels.index("under_radial")] += amp * np.sin(2 * np.pi * L * fr * t + L)
    return x.astype(np.float32)


ch = sc.channels.index("under_radial")
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8), sharey=True)
for ax, fr in zip(axes, (12.0, 30.0, 60.0)):
    x = synthetic(fr)
    for name in sc.candidates:
        prof = recording_representation(x, fr, pre, name)["order"][:, ch].mean(axis=(0, 2))
        ax.semilogx(orders, prof, lw=1, label=name)
    for L in (0.5, 1, 2, 3):
        ax.axvline(L, color="grey", lw=0.5, ls=":")
    ax.set(xlim=(0.4, 5), title=f"synthetic, {fr:g} Hz", xlabel="order (× shaft speed)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("amplitude (dB re 1 V)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "04_resolution_synthetic.png")

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates the measured resolution: the −3 dB width of the 1× peak (in orders) and the depth of the valley between 0.5× and 1× and between 1× and 2× (a valley of at least 3 dB counts as separated), for synthetic signals at 12, 30 and 60 Hz and for real subset recordings near 12 and 30 Hz.</div>

In [ ]:
res = pd.read_csv(RES / "resolution.csv")
res["speed"] = np.where(res.source == "synthetic", res.speed_hz.astype(int).astype(str) + " Hz",
                        np.where(res.speed_hz < 20, "real ~13–17 Hz", "real ~29–30 Hz"))
cols = ["1x_fwhm_orders", "valley_0.5x_1x_db", "valley_1x_2x_db"]
res.groupby(["candidate", "speed"])[cols].mean().round(3).unstack("candidate")

Real subset recordings near 12 Hz do not exist in this 70-recording subset: the slowest shaft-fault recordings chosen are
at 13–17 Hz. A real 0.5× line is rarely present, so the 0.5×–1× valley on real data mostly measures noise between them.

## 3. Speed-related sharpness
Even when every row has data, a representation can carry speed through how sharp its rows are. Per recording: each order
row's local contrast (its level minus the median of rows ±3, averaged over time and channels) → predict the speed tercile
with logistic regression, 5-fold CV over recordings (windows of a recording stay together), against 20 label permutations.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the speed-sharpness diagnostic and shows, per candidate, how well the row sharpness predicts the speed tercile compared with shuffled labels. Higher means the representation carries more speed information.</div>

In [ ]:
sharp = json.loads((RES / "sharpness.json").read_text())
pd.DataFrame(sharp).T.round(3)

All three carry speed information well above the permutation baseline; A the most, C the least. Part of this is
physical (vibration levels and spectra change with speed), so a non-chance score is expected; the ranking shows how much
the representation adds on top through resolution.

## 4. Gallery: one training recording per class
Mean over the six accelerometer channels, window 4 (the middle of the recording), each candidate.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For each class, takes one subset recording and shows each candidate's order-spectrogram (the average over the six accelerometers) for its middle window, on the common log order axis.</div>

In [ ]:
names = list(sc.candidates)
z0 = subs[names[0]]
pick = {c: int(np.flatnonzero(z0["class10"] == c)[0]) for c in CLASS10}
fig, axes = plt.subplots(len(names), len(CLASS10), figsize=(22, 7.5), sharey=True)
for r, name in enumerate(names):
    z = subs[name]
    for c_i, cls in enumerate(CLASS10):
        i = int(np.flatnonzero(z["rec_id"] == z0["rec_id"][pick[cls]])[0])
        img = z["order"][i, 4].astype(np.float32).mean(axis=0)
        ax = axes[r, c_i]
        ax.imshow(img, aspect="auto", origin="lower", cmap="magma", vmin=np.percentile(img, 5), vmax=img.max(),
                  extent=[0, 1, 0, len(orders)])
        ticks = [np.argmin(np.abs(orders - v)) for v in (0.5, 1, 3, 10, 30, 100)]
        ax.set_yticks(ticks, ["0.5", "1", "3", "10", "30", "100"])
        if r == 0:
            ax.set_title(f"{cls}\n{float(z['fr_hz'][i]):.0f} Hz", fontsize=8)
        if c_i == 0:
            ax.set_ylabel(f"{name}\norder", fontsize=8)
        ax.set_xticks([])
fig.tight_layout()
fig.savefig(FIG / "04_gallery.png")

## 5. Microphone log-Mel, aligned with each candidate
128 HTK Mel bins from 20 Hz to 10 kHz, 2,048-sample Hann frames (41 ms) centred at **the same times** as the candidate's
frames, `10·log10(power + 1e-12)`, on the cleaned microphone. One recording, candidate A and C (T = 25) and B (T = 10).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows, for one subset recording, the order-spectrogram of one accelerometer and the microphone log-Mel for the same window, under each candidate, with frame centre times marked, and checks that the two share their frame times.</div>

In [ ]:
from ml.spectro import frame_centres

rid = str(z0["rec_id"][pick["imbalance"]])
fig, axes = plt.subplots(2, len(names), figsize=(16, 6))
for c_i, name in enumerate(names):
    z = subs[name]
    i = int(np.flatnonzero(z["rec_id"] == rid)[0])
    T = sc.candidates[name].frames_per_window
    centres_ms = (frame_centres(pre.n_rows, pre, T)[4] - frame_centres(pre.n_rows, pre, T)[4][0]) / pre.fs_hz * 1e3
    axes[0, c_i].imshow(z["order"][i, 4, sc.channels.index("under_radial")].astype(np.float32), aspect="auto",
                        origin="lower", cmap="magma")
    axes[0, c_i].set(title=f"{name}: under_radial order-spectrogram", ylabel="order bin")
    axes[1, c_i].imshow(z["mel"][i, 4].astype(np.float32), aspect="auto", origin="lower", cmap="viridis")
    axes[1, c_i].set(title=f"mic log-Mel, same {T} frame times", ylabel="Mel bin", xlabel="frame")
    assert z["order"].shape[-1] == z["mel"].shape[-1] == T
fig.suptitle(rid)
fig.tight_layout()
fig.savefig(FIG / "04_mel_aligned.png")
print("both images of every candidate have T frames centred at the same sample indices (ml.spectro.frame_centres)")

### The 10 kHz cap on the microphone
Measured on the 70 subset recordings (before = bursts removed and band-stop applied, without the low-pass).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the cap verification and shows, over the subset recordings, the measured attenuation of microphone power above 10.2 kHz, the residual level relative to the pass band, the change in the pass band, and the measured edge effect at each recording end.</div>

In [ ]:
cap = pd.read_csv(RES / "mic_cap.csv")
cap.drop(columns="rec_id").describe().loc[["min", "50%", "max"]].round(3).T

The cap is not "no energy above 10 kHz": power above 10.2 kHz is reduced by 106.6–111.0 dB and sits 108–112 dB below
the pass band; the pass band is unchanged; the edge effect is at most 3 ms (−40 dB) and 4 ms (−60 dB).

### One burst file, before and after cleaning
A development-pool **training** recording flagged by the burst threshold: its microphone log-Mel without and with burst
removal (same window).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Takes one flagged training recording from the development pool, computes the microphone log-Mel with burst removal switched off and on (everything else identical), and shows both for the window with the largest burst, plus their difference.</div>

In [ ]:
from ml.preprocess import clean_recording
from ml.spectro import log_mel
from ml.splits import development_pool

dc = cfg.development
pool = development_pool(meta, dc.allowed_closed, dc.reserved_columns)
flagged = pool[(pool.split_closed == "train") & (pool.peakz_mic > pre.mic.burst_z)].sort_values("peakz_mic")
rid_b = flagged.rec_id.iloc[len(flagged) // 2]
x = np.load(data_dir() / "raw_npy" / f"{rid_b}.npy")
no_burst_removal = pre.model_copy(update={"mic": pre.mic.model_copy(update={"burst_z": 1e12})})
mic_i = pre.channels.index("mic")
raw_m = clean_recording(x, no_burst_removal)[0][:, mic_i]
cln_m = clean_recording(x, pre)[0][:, mic_i]
w_i = int(np.argmax([np.abs(raw_m[s:s + pre.fs_hz]).max() for s in range(0, 4 * pre.fs_hz + 1, pre.fs_hz // 2)]))
a, b = log_mel(raw_m, pre, 25)[w_i], log_mel(cln_m, pre, 25)[w_i]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, img, title in zip(axes, (a, b, a - b), ("without burst removal", "with burst removal", "difference (dB)")):
    im = ax.imshow(img, aspect="auto", origin="lower", cmap="viridis" if "diff" not in title else "RdBu_r")
    ax.set(title=f"{title}", xlabel="frame", ylabel="Mel bin")
    fig.colorbar(im, ax=ax)
fig.suptitle(f"{rid_b}, window {w_i}")
fig.tight_layout()
fig.savefig(FIG / "04_burst_before_after.png")
print(f"{rid_b}: peak robust z {flagged.set_index('rec_id').loc[rid_b, 'peakz_mic']:.0f}; largest per-frame change "
      f"{np.abs(a - b).max():.1f} dB; frames changed by > 1 dB in any Mel bin: {int(((np.abs(a - b) > 1).any(axis=0)).sum())} of 25")

### Burst-residue diagnostic
In the development pool's affected folders: flagged-and-cleaned vs unflagged recordings.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the burst-residue counts: per affected folder, how many recordings are flagged and unflagged, their speed ranges, and how many of each fall in the speed range the two groups share.</div>

In [ ]:
mic_d = json.loads((RES / "mic_diagnostics.json").read_text())
display(pd.DataFrame(mic_d["burst_residue_counts"]).set_index("folder"))
print(mic_d["burst_residue_verdict"])

**No conclusion is possible.** Within every affected folder, flagged and unflagged recordings do not overlap in speed:
in imbalance/30g the unflagged ones run 12.9–21.1 Hz and the flagged 22.1–39.8 Hz; in underhang_cage/20g, up to 19.0 Hz vs
from 19.9 Hz; imbalance/20g has one flagged recording. There are no comparable controls, and any flagged-vs-unflagged
comparison would confound the bursts with speed. The observation itself (flags start near 20 Hz within these sessions) is
worth recording.

## 6. The candidates side by side (for the choice)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates, per candidate, the would-be stacked shape, the float16 size, the measured time per recording and the projected full build time, the 1× peak width, whether 0.5× and 1× separate, and the speed-sharpness score.</div>

In [ ]:
rows = []
for name, c in summary["candidates"].items():
    rows.append({"candidate": name, "T": c["T"], "stacked shape": " × ".join(map(str, c["stacked_shape"])),
                 "float16 GB": c["float16_gb"], "s / recording / core": c["seconds_per_recording_per_core"],
                 "projected full build (min, 12 workers)": c["projected_full_build_min"],
                 "1× width 12/30/60 Hz (orders)": " / ".join(str(v) for v in c["fwhm_1x_orders_synthetic"].values()),
                 "0.5× vs 1× separated 12/30/60 Hz": " / ".join("yes" if v else "no" for v in c["separates_0p5x_1x_synthetic"].values()),
                 "speed tercile from sharpness (perm.)": f"{c['speed_tercile_from_sharpness']['macro_f1']:.3f} "
                                                         f"({c['speed_tercile_from_sharpness']['perm_mean']:.3f})"})
print("cache-key inputs:", summary["candidates"]["A_stft8192"]["cache_key_inputs"])
pd.DataFrame(rows).set_index("candidate")

## Findings (prototypes; nothing frozen)
- **Resolution at low orders.** −3 dB width of the 1× peak, synthetic 12 / 30 / 60 Hz: A 0.905 / 0.265 / 0.135 orders, B 0.135 / 0.045 / 0.045 (the log-axis bin width near 1×), C 0.135 at every speed. **A cannot separate 0.5× from 1× at 12 Hz** (0.9 dB valley); B and C can at every speed tested. On real subset recordings at ~13–17 Hz the 1× width is A 0.62, B 0.12, C 0.135 orders.
- **Speed carried by sharpness.** Speed tercile predicted from per-row sharpness (recordings kept together): A 0.943, B 0.761, C 0.668, against a permutation mean of ~0.32. All carry speed; part of it is physical, and A adds the most through resolution.
- **Costs.** T = 25 (A, C) or 10 (B); stacked float16 0.79 GB (A, C) or 0.32 GB (B); 0.36 / 0.58 / 1.18 s per recording per core, so a full build would take ~1.0 / 1.6 / 3.2 min on 12 workers. B pays in time resolution (0.655 s frames, 85 % overlap); C's time resolution varies with speed (a frame is 8 revolutions: 0.13 s at 62 Hz, 0.68 s at 11.8 Hz).
- **Microphone.** The log-Mel frames share each candidate's frame times. The 10 kHz cap reduces microphone power above 10.2 kHz by 106.6–111.0 dB (residual 108–112 dB below the pass band; not "no energy"), with ≤ 4 ms edge effects. On a flagged training recording, burst removal changes the log-Mel by up to 54.6 dB in 24 of 25 frames of the worst window.
- **Burst residue: no conclusion.** In every affected development folder, flagged and unflagged recordings do not overlap in speed (flags start near 20 Hz), so no comparable controls exist.
- **Mapping detail:** spectra are mapped onto order bins by the maximum over each bin's span; point sampling lost up to 1.9 dB on sharp peaks (B at 60 Hz).

## Outputs written
- `results/figures/04_resolution_synthetic.png`, `04_gallery.png`, `04_mel_aligned.png`, `04_burst_before_after.png`
- `results/representations/`: `resolution.csv`, `sharpness.json`, `mic_cap.csv`, `burst_residue_counts.csv`, `mic_diagnostics.json`, `candidates_summary.json`
- Subset caches `data/cache/08_subset_<candidate>_<sha>.npz` (not committed). No full cache.


---
# Part 2 (Stage 2C-2): the speed-risk check and the frozen cache

## 7. Speed-risk check, C vs B
Rule committed before running (`scripts/09_speed_risk.py`): group (ii) = train ∩ development-pool recordings whose window
speeds vary by more than 1 %; group (iii) = two nearest-speed clean recordings of the same class for each; synthetic
stress = a corrupted tachometer that forces the fallback to the supplied speed, offset by 0, ±0.5, ±1, ±2 %. C passes
only if, on group (ii) and on ±1 %, its median 1× position error is ≤ 1 order bin and its median 1× width ≤ 1.5 × its own
reference, and it is no more than 1 bin / 1.5 × (width ratio) worse than B.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the speed-risk results and tabulates, for C and B, the median 1× and 2× position errors (in order bins), the median 1× width and its ratio to each candidate's own reference, per group and per speed offset, with the pass checks.</div>

In [ ]:
sr = json.loads((RES / "speed_risk.json").read_text())
print("counts:", sr["counts"], "| order bin at 1× ≈", round(sr["order_bin_at_1x_orders"], 4), "(log step)")
rows = []
for c, e in sr["candidates"].items():
    rows.append({"candidate": c, "case": "group (ii)", **{k: round(v, 3) for k, v in e["group_ii"].items()}})
    rows.append({"candidate": c, "case": "group (iii) reference", "median_x1_err_bins": round(e["group_iii"]["median_x1_err_bins"], 3),
                 "median_x1_width": round(e["group_iii"]["median_x1_width"], 3), "width_ratio": 1.0})
    for off, v in e["synthetic"].items():
        rows.append({"candidate": c, "case": f"synthetic fallback {off}", **{k: round(x, 3) for k, x in v.items()}})
display(pd.DataFrame(rows).set_index(["candidate", "case"]))
print("checks:", json.dumps(sr["checks"]), "→ C passes:", sr["C_passes"])

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the example figure: the frame-averaged order profile near 0.5–3× for one recording of each group, for C (solid) and B (dashed); for the synthetic example, the clean version and the fallback with +1 % and +2 % speed error.</div>

In [ ]:
img = plt.imread(FIG / "09_speed_risk_examples.png")
plt.figure(figsize=(17, 4))
plt.imshow(img)
plt.axis("off")

**C passes** every pre-set check. This is **a limited engineering check**. Widths were compared relative to each candidate's own clean reference, as committed before running. It shows no degradation of one bin or more under ±1 % speed error; ±2 % degrades 2× by about one bin. Real fallback recordings remain untested (none in the dev pool), and small errors can be hidden by the 4.6 %-wide order bins. C is frozen (`representation.frozen`).

## 8. The frozen cache
All 1,951 recordings → `data/representations/C_order8rev_<key>.npy` (17,559 windows × 7 × 128 × 25, float16): six
accelerometer order-spectrograms and the time-aligned microphone log-Mel. The key covers the full preprocess config, the
candidate, the representation version and every recording's checksum and supplied speed; loading recomputes it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the cache through its key check and reports its shape, size, that every value is finite, and that its rows line up with the capped feature table; then the serving latency of one recording.</div>

In [ ]:
from ml.cache import load_features
from ml.datasets import load_representation, norm_stats

X, index = load_representation(data_dir() / "representations", cfg, meta)
finite = all(np.isfinite(np.asarray(X[i:i + 2048], dtype=np.float32)).all() for i in range(0, X.shape[0], 2048))
feat_path = max((data_dir() / "features").glob("r1cap10_*.parquet"), key=lambda q: q.stat().st_mtime)
ft = load_features(feat_path, meta, pre).sort_values(["rec_id", "window_idx"]).reset_index(drop=True)
aligned = bool((ft[["rec_id", "window_idx"]].to_numpy() == index[["rec_id", "window_idx"]].to_numpy()).all())
print(f"shape {X.shape} {X.dtype}, {X.nbytes / 1e9:.2f} GB; all finite: {finite}; rows aligned with {feat_path.name}: {aligned}")
print("serving latency, one recording (CSV load + representation, one process): median 0.51 s, max 1.02 s (5 train recordings)")

### Gallery from the cache
One training recording per class, window 4: the mean of the six accelerometer order-spectrograms and the microphone log-Mel.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Reads one training recording per class from the cache and shows, for its middle window, the average accelerometer order-spectrogram (top) and the microphone log-Mel (bottom).</div>

In [ ]:
train_ids = meta[meta.split_closed == "train"].groupby("class10").rec_id.first()
fig, axes = plt.subplots(2, len(CLASS10), figsize=(22, 5.5))
for j, cls in enumerate(CLASS10):
    rows_ = np.flatnonzero(index.rec_id.to_numpy() == train_ids[cls])
    w = np.asarray(X[rows_[4]], dtype=np.float32)
    for r, (img, ttl) in enumerate([(w[:6].mean(axis=0), "order (mean of 6 accel.)"), (w[6], "mic log-Mel")]):
        axes[r, j].imshow(img, aspect="auto", origin="lower", cmap="magma" if r == 0 else "viridis",
                          vmin=np.percentile(img, 5), vmax=img.max())
        axes[r, j].set_xticks([])
        axes[r, j].set_yticks([])
    axes[0, j].set_title(f"{cls}\n{meta.set_index('rec_id').loc[train_ids[cls], 'fr_measured_hz']:.0f} Hz", fontsize=8)
axes[0, 0].set_ylabel("order bins")
axes[1, 0].set_ylabel("Mel bins")
fig.tight_layout()
fig.savefig(FIG / "04_gallery_cache.png")

### Per-channel value ranges
Over the closed-set **training** windows (the statistics that `ml.datasets.norm_stats` computes at training time; they are not stored in the cache).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Computes, for each of the 7 channels, the mean and standard deviation over the closed-set training windows with norm_stats, and the 1st, 50th and 99th percentiles from a sample of 2,000 of those windows.</div>

In [ ]:
st = norm_stats("closed", X, index, meta)
from ml.datasets import track_train_mask

rows_tr = np.flatnonzero(track_train_mask(index, meta, "closed"))
samp = np.asarray(X[np.sort(np.random.default_rng(0).choice(rows_tr, 2000, replace=False))], dtype=np.float32)
ch_names = list(sc.channels) + ["mic log-Mel"]
pd.DataFrame({"mean (dB)": st["mean"], "std (dB)": st["std"],
              **{f"p{q} (dB)": np.percentile(samp, q, axis=(0, 2, 3)) for q in (1, 50, 99)}}, index=ch_names).round(1)

## Findings (part 2)
- **The speed-risk check (a limited engineering check) passes for C** on group (ii) (12 recordings, 24 clean references) and on the synthetic ±1 % fallback (20 recordings): median 1× error 0.434 bins (the axis offset, same as its references and as B), 1× width 0.135 orders (ratio 1.0). At ±2 % the 2× peak shifts about one bin and C's 1× width grows to 1.30 × at +2 % (sensitivity only). Real fallback recordings remain untested (none in the dev pool), and the 4.6 %-wide order bins hide errors below about half a bin.
- **C is frozen and cached:** 17,559 × 7 × 128 × 25 float16 (0.79 GB) in 161 s on 12 workers; all values finite; rows aligned with the capped feature table; key verified on load; batch and serving outputs are bit-identical; one recording takes 0.51 s (median) on one CPU process.
- **Scales differ by channel** (training means from −52 dB for the microphone log-Mel to −29 dB for under_axial), so per-channel normalisation, fitted per track on training windows only, is needed before any model.

## Outputs written (part 2)
- `results/representations/speed_risk.json`, `speed_risk_windows.csv`; `results/figures/09_speed_risk_examples.png`, `04_gallery_cache.png`
- `data/representations/C_order8rev_<key>.npy|.index.parquet|.json` (not committed)